In [24]:
import pandas as pd
import numpy as np
from Utilities import *
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import joblib

def Prepare_Data(data):
    data.dropna(inplace= True)
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index(data['Date'])
    ### Adding indicators to help the model catch trend and seasonality.
    data['SMA'] = SMA(data, 12)
    data['EMA'] = EMA(data, 12)
    data['MACD'] = MACD(data, 12, 26)
    data['RSI'] = RSI(data, 12)
    data['upper_band'], data['lower_band'] = Bollinger_Bands(data, 12)
    data.dropna(inplace=True)

    target = data['Close'].shift(-1)[:-1]
    Features = data.drop(columns = ['Close', 'Date'], axis = 1)

    Feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()


    TS = target_scaler.fit(target.values.reshape(-1,1))
    FS = Feature_scaler.fit(Features)
    F_scaled = pd.DataFrame(FS.transform(Features.values), columns = Features.columns)
    T_scaled = pd.DataFrame(TS.transform(target.values.reshape(-1,1)))
    joblib.dump(TS, "TS.save")
    joblib.dump(FS, "FS.save")

    X, y = createdataset(F_scaled, T_scaled, 1)


    Xtr = X[:int(X.shape[0]*0.8)]
    Xts = X[int(X.shape[0]*0.8):]
    ytr = y[:int(y.shape[0]*0.8)]
    yts = y[int(y.shape[0]*0.8):]

    return Xtr, ytr, Xts, yts


class DeepARModel(nn.Module):
    def __init__(self, input_size, hidden_size=50):
        super(DeepARModel, self).__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True )
        self.elu1 = nn.ELU(alpha=0.5)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.elu2 = nn.ELU(alpha=0.5)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.elu1(x)
        x, _ = self.lstm2(x)
        x = self.elu2(x)
        x = self.fc(x)
        return x

def Training(Xtr, ytr, Xts, yts):

# ... (previous code, including model definition)

# Convert data to PyTorch tensors
    Xtr_tensor = torch.FloatTensor(Xtr)
    ytr_tensor = torch.FloatTensor(ytr).squeeze(-1)  # Remove the last dimension
    Xts_tensor = torch.FloatTensor(Xts)
    yts_tensor = torch.FloatTensor(yts).squeeze(-1)

    # Create DataLoader for training data
    train_dataset = TensorDataset(Xtr_tensor, ytr_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Assuming X.shape[2] is the number of features
    input_size = Xtr.shape[2]
    model = DeepARModel(input_size)

    # Define loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())

    # Training loop
    num_epochs = 100
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Print average loss for the epoch
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_predictions = model(Xts_tensor).squeeze(-1)
        test_loss = criterion(test_predictions, yts_tensor)
        print(f'Test Loss: {test_loss.item():.4f}')

    # Save the model
    torch.save(model.state_dict(), 'deepar_model.pth')

In [25]:
data = pd.read_csv("NSEI.csv")
data

,Date,Open,High,Low,Close,Adj Close,Volume
0,2023-09-05,19564.650391,19587.050781,19525.750000,19574.900391,19574.900391,256800
1,2023-09-06,19581.199219,19636.449219,19491.500000,19611.050781,19611.050781,287600
2,2023-09-07,19598.650391,19737.000000,19550.050781,19727.050781,19727.050781,304900
3,2023-09-08,19774.800781,19867.150391,19727.050781,19819.949219,19819.949219,288100
4,2023-09-11,19890.000000,20008.150391,19865.349609,19996.349609,19996.349609,248800
...,...,...,...,...,...,...,...
240,2024-08-30,25249.699219,25268.349609,25199.400391,25235.900391,25235.900391,638200
241,2024-09-02,25333.599609,25333.650391,25235.500000,25278.699219,25278.699219,222800
242,2024-09-03,25313.400391,25321.699219,25235.800781,25279.849609,25279.849609,212100
243,2024-09-04,25089.949219,25216.000000,25083.800781,25198.699219,25198.699219,253800


In [26]:
Xtr, ytr, Xts, yts = Prepare_Data(data)

c:\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [27]:
Xtr.shape

(185, 1, 11)

In [28]:
Training(Xtr, ytr, Xts, yts)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([32, 1])) that is different to the input size (torch.Size([32, 1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([25, 1])) that is different to the input size (torch.Size([25, 1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/100], Loss: 0.1712
Epoch [2/100], Loss: 0.1504
Epoch [3/100], Loss: 0.1314
Epoch [4/100], Loss: 0.1109
Epoch [5/100], Loss: 0.0894
Epoch [6/100], Loss: 0.0674
Epoch [7/100], Loss: 0.0507
Epoch [8/100], Loss: 0.0440
Epoch [9/100], Loss: 0.0437
Epoch [10/100], Loss: 0.0450
Epoch [11/100], Loss: 0.0441
Epoch [12/100], Loss: 0.0429
Epoch [13/100], Loss: 0.0431
Epoch [14/100], Loss: 0.0416
Epoch [15/100], Loss: 0.0410
Epoch [16/100], Loss: 0.0420
Epoch [17/100], Loss: 0.0411
Epoch [18/100], Loss: 0.0411
Epoch [19/100], Loss: 0.0409
Epoch [20/100], Loss: 0.0414
Epoch [21/100], Loss: 0.0410
Epoch [22/100], Loss: 0.0407
Epoch [23/100], Loss: 0.0415
Epoch [24/100], Loss: 0.0398
Epoch [25/100], Loss: 0.0412
Epoch [26/100], Loss: 0.0405
Epoch [27/100], Loss: 0.0406
Epoch [28/100], Loss: 0.0402
Epoch [29/100], Loss: 0.0401
Epoch [30/100], Loss: 0.0410
Epoch [31/100], Loss: 0.0404
Epoch [32/100], Loss: 0.0401
Epoch [33/100], Loss: 0.0402
Epoch [34/100], Loss: 0.0401
Epoch [35/100], Loss: 0

In [29]:
def load_model_and_scalers(model_path, feature_scaler_path, target_scaler_path, input_size):
    # Load the model
    model = DeepARModel(input_size)
    model.load_state_dict(torch.load(model_path))
    model.eval()  # Set the model to evaluation mode

    # Load the scalers
    FS = joblib.load(feature_scaler_path)
    TS = joblib.load(target_scaler_path)

    return model, FS, TS

def add_indicators(data):
    # Assuming these functions are defined in your Utilities module
    data['SMA'] = SMA(data, 12)
    data['EMA'] = EMA(data, 12)
    data['MACD'] = MACD(data, 12, 26)
    data['RSI'] = RSI(data, 12)
    data['upper_band'], data['lower_band'] = Bollinger_Bands(data, 12)
    return data



In [30]:
def predict_next_day(model, Feature_scaler, target_scaler, latest_data):
    # Ensure the model is in evaluation mode
    model.eval()

    # Convert latest_data to DataFrame if it's not already
    if not isinstance(latest_data, pd.DataFrame):
        latest_data_df = pd.DataFrame([latest_data])
    else:
        latest_data_df = latest_data.copy()

    print("Shape of latest_data_df:", latest_data_df.shape)
    print("Columns of latest_data_df:", latest_data_df.columns)

    # Add indicators to the latest data
    latest_data_with_indicators = add_indicators(latest_data_df)
    
    print("Shape after adding indicators:", latest_data_with_indicators.shape)
    print("Columns after adding indicators:", latest_data_with_indicators.columns)

    # Instead of dropping NaN values, fill them with the last known value
    latest_data_with_indicators = latest_data_with_indicators.fillna(value = 0)
    print("Shape after filling NaN:", latest_data_with_indicators.shape)

    # Use only the features used during training
    features_to_use = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA', 'EMA', 'MACD', 'RSI', 'upper_band', 'lower_band']
    
    # Check if all required features are present
    missing_features = set(features_to_use) - set(latest_data_with_indicators.columns)
    if missing_features:
        raise ValueError(f"Missing features: {missing_features}")

    latest_data_array = latest_data_with_indicators[features_to_use].values

    print("Shape of latest_data_array:", latest_data_array.shape)

    if latest_data_array.shape[0] == 0:
        raise ValueError("No data available for prediction after processing")

    # Scale the data
    scaled_data = Feature_scaler.transform(latest_data_array)
    
    # Convert to PyTorch tensor and reshape for the model
    input_tensor = torch.FloatTensor(scaled_data).unsqueeze(0)  # Shape: (1, num_features)

    # Make prediction
    with torch.no_grad():
        prediction = model(input_tensor).squeeze()

    # Inverse transform the prediction
    unscaled_prediction = target_scaler.inverse_transform(prediction.numpy().reshape(-1, 1))

    return unscaled_prediction[0][0]

# Usage


In [31]:
data = add_indicators(data)
latest_data = data.iloc[-1]

# Predict the next day's price
latest_data

Date          2024-09-05 00:00:00
Open                      25250.5
High                 25275.449219
Low                  25233.099609
Close                     25241.5
Adj Close                 25241.5
Volume                          0
SMA                  25072.678874
EMA                  25060.408443
MACD                    238.92517
RSI                      2.166476
upper_band           25450.196099
lower_band           24695.161649
Name: 244, dtype: object

In [32]:
model, FS , TS  = load_model_and_scalers("deepar_model.pth", "FS.save", "TS.save", 11)

C:\Users\user\AppData\Local\Temp\ipykernel_26392\1007221217.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


In [33]:
# Usage
try:
    next_day_price = predict_next_day(model, FS, TS, latest_data)
    print(f"Predicted price for the next day: {next_day_price:.2f}")

    # Compare with the last known price
    last_known_price = data['Close'].iloc[-1]
    print(f"Last known price: {last_known_price:.2f}")
    print(f"Predicted change: {next_day_price - last_known_price:.2f}")
except Exception as e:
    print("Error occurred:", str(e))
    print("latest_data:", latest_data)

Shape of latest_data_df: (1, 13)
Columns of latest_data_df: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'SMA',
       'EMA', 'MACD', 'RSI', 'upper_band', 'lower_band'],
      dtype='object')
Shape after adding indicators: (1, 13)
Columns after adding indicators: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'SMA',
       'EMA', 'MACD', 'RSI', 'upper_band', 'lower_band'],
      dtype='object')
Shape after filling NaN: (1, 13)
Shape of latest_data_array: (1, 11)
Predicted price for the next day: 20762.12
Last known price: 25241.50
Predicted change: -4479.38


c:\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
